# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")
df.head()

## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [ ]:
# Variáveis independentes
X = df.drop(columns=[TARGET])

# Variável Alvo
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

*Modelo KNN*

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

accuracy = knn.score(X_test, y_test)

# Erro médio absoluto
MAE = mean_absolute_error(y_test, y_pred_knn)

r2 = r2_score(y_test, y_pred_knn) # qto mais próximo de 1, melhor

print('MAE',MAE)
print('r²',r2)

print("Acurácia: {:.2f}".format(round(accuracy,4)))

In [ ]:
modelos_desbalanceados = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=3)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
}


modelos_balanceados = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=3)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
}

## 3. Validação cruzada

In [ ]:
from sklearn.model_selection import cross_val_score

print("Modelos desbalanceados")
for nome, modelo in modelos_desbalanceados.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')

print("\nModelos balanceados")
for nome, modelo in modelos_balanceados.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')
    




## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_